# 🖼️ Lesson 4.3 — Convolutional Neural Networks: How Machines See

**AI Crash Course for Absolute Beginners**

CNNs are specialised for grid data like images. In this notebook:
- See what convolution actually does to an image
- Build a CNN in PyTorch and train it on MNIST
- Visualise feature maps (what the network 'sees')
- Add batch normalisation and max-pooling

> Run each cell with **Shift + Enter**. GPU recommended but not required.

## ⚠️ GPU Recommended — Enable Before Running

Training a CNN on MNIST takes **10–15 minutes on CPU** but only **2–3 minutes on GPU**.

**Enable GPU now (free):**
1. Click **Runtime** in the top menu
2. Select **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save**, then run from the top

> You only need to do this once per session. The cell below will confirm your device.

In [ ]:
!pip install torch torchvision matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---
## Part 1 — What Does Convolution Actually Do?

In [ ]:
from scipy.ndimage import convolve
from skimage import data as skdata

# Load a grayscale image (comes with scikit-image)
image = skdata.camera().astype(float) / 255.0   # 512x512 photographer

# Three classic kernels
kernels = {
    "Edge detection\n(Sobel-X)": np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    "Blur\n(Gaussian 3x3)"     : np.ones((3,3)) / 9,
    "Sharpen"                  : np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    filtered = convolve(image, kernel)
    filtered = np.clip(filtered, 0, 1)
    ax.imshow(filtered, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.suptitle("Convolution with Different Kernels", fontsize=13)
plt.tight_layout()
plt.show()
print("In a CNN, the network learns these kernels automatically from data.")

---
## Part 2 — Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST mean and std
])

train_data = datasets.MNIST(root="data", train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256)

print(f"Training images : {len(train_data):,}")
print(f"Test images     : {len(test_data):,}")
print(f"Image shape     : {train_data[0][0].shape}  (channels x H x W)")

# Preview samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(label, fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST Samples", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 3 — Build the CNN

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional feature extractor
        self.features = nn.Sequential(
            # Block 1: 1x28x28 -> 32x26x26 -> 32x13x13
            nn.Conv2d(1, 32, kernel_size=3),   # 32 filters, 3x3 kernel
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                   # halve spatial dims
            # Block 2: 32x13x13 -> 64x11x11 -> 64x5x5
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)          # 10 digit classes
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = MnistCNN().to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## Part 4 — Train the CNN

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

EPOCHS = 8
train_accs, test_accs = [], []

for epoch in range(EPOCHS):
    # Training
    model.train()
    correct = total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        preds = model(X_batch).argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total   += len(y_batch)
    train_accs.append(correct / total)

    # Evaluation
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    test_accs.append(correct / total)
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS}  train={train_accs[-1]:.3f}  test={test_accs[-1]:.3f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS+1), train_accs, label="Train", marker="o", color="#1a6bc8")
plt.plot(range(1, EPOCHS+1), test_accs,  label="Test",  marker="s", color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"CNN on MNIST — Final test accuracy: {test_accs[-1]:.1%}")
plt.legend()
plt.tight_layout()
plt.show()

---
## Part 5 — Visualise Feature Maps

In [ ]:
# Pick one test image
sample_img, sample_label = test_data[7]
sample_img_batch = sample_img.unsqueeze(0).to(device)

# Hook to capture intermediate activations
activations = {}
def hook_fn(name):
    def fn(module, input, output):
        activations[name] = output.detach().cpu()
    return fn

model.features[0].register_forward_hook(hook_fn("conv1"))
model.features[4].register_forward_hook(hook_fn("conv2"))

model.eval()
with torch.no_grad():
    out = model(sample_img_batch)
    pred = out.argmax(dim=1).item()

print(f"True label: {sample_label}  |  Predicted: {pred}")

# Plot first 16 feature maps from conv1
feature_maps = activations["conv1"].squeeze()
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
axes[0][0].imshow(sample_img.squeeze(), cmap="gray")
axes[0][0].set_title(f"Input (digit {sample_label})")
axes[0][0].axis("off")

for i, ax in enumerate(axes.flatten()[1:15]):
    ax.imshow(feature_maps[i], cmap="viridis")
    ax.set_title(f"Filter {i+1}")
    ax.axis("off")

plt.suptitle("Conv Layer 1 Feature Maps", fontsize=13)
plt.tight_layout()
plt.show()

---
## Challenge Exercises

1. **CIFAR-10**: Replace MNIST with `datasets.CIFAR10`. The images are 32x32x3. Adjust the first Conv layer to accept 3 channels (`nn.Conv2d(3, 32, 3)`).
2. **More filters**: Double the filter count to 64 in the first block. How many more parameters does this add?
3. **Wrong predictions**: Find test images the model got wrong. Do they look ambiguous to you too?

---
**Next lesson:** [Lesson 4.4 — Transformers and Attention](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.4-transformers.ipynb)